---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"
---

In [1]:
knitr::opts_chunk$set(echo = TRUE)

## Load Required Libraries

In [2]:
options(verbose = FALSE)
options(warn = -1)

In [3]:
suppressPackageStartupMessages({
  library(rprojroot)
  # library(conflicted)
  # library(tidyverse)
  library(data.table)
  library(here)
  library(tictoc)
  library(stringr)
  library(stringi)
  library(lubridate)
  library(docstring)
  library(profvis)
  library(hash)
  # library(foreach)
  # library(doParallel)
  # library(parallel)
  library(future)
  library(future.apply)
  library(knitr)
})
print("Packages loaded successfully.")
setwd("C:/drg-pipeline")
here::i_am("drg-pipeline/.here")

[1] "Packages loaded successfully."
[1] "Packages loaded successfully."


here() starts at C:/
here() starts at C:/


In [4]:
options(warn = 1)

## Set Parameters

### Year to Load & Version

In [5]:
year_to_load <- "2018"
version <- "v2"

### Parameters

In [6]:
sample_size <- 250 * 1e3
seed <- 123

drop_cols <- c(paste0("ICDCODE", 13:14), "ICCODED15", paste0("ICDCODE", 16:170))
icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)


In [7]:
to_read <- FALSE
to_sample <- TRUE
to_write <- TRUE
to_group <- TRUE
to_filter <- FALSE # unused
to_profvis <- FALSE
to_chunk <- FALSE
to_view_checks <- TRUE


In [8]:
set.seed(seed)

options(future.globals.maxSize = 1024 * 1024 ^ 2)

global_seed <- seed #for parallelized operations


## Source Data Formats

In [9]:
source(here("drg-pipeline", "data-cleaning", "r_scripts", "data-formats.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "file-paths.R"))

## Source Functions

In [10]:
source(here("drg-pipeline", "data-cleaning", "r_scripts", "general-functions.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "main-functions.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "icd-functions.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("drg-pipeline", "data-cleaning", "r_scripts", "grouper-functions.R"))

## Load Mapping Data

In [11]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
                  select = c("rvs", "icd9cm"))
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
                  by.x = "icd9cm", by.y = "CODE", all.x = TRUE)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)

## Read Data

### Reading Data

In [12]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}

[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file exists. Reading the sampled file..."


[1] "Sampled file matches sample size."
[1] "Sampled file matches sample size."


In [13]:
options(warn = 1)

## Data Processing

### Data Cleaning

#### Not chunking

In [14]:
if (!to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      dt <- clean_data(dt)
    })
  } else {
    dt <- clean_data(dt)
  }
}

 [1] "id_series"           "id_pin"              "date_adm"            "time_adm"            "date_dis"            "time_dis"            "date_rec"           
 [8] "date_ref"            "date_check"          "id_hci"              "id_hcp"              "clin_outpatient"     "clin_emergency"      "pat_type"           
[15] "clin_acc"            "pat_rel"             "pat_bdate"           "pat_age"             "pat_sex"             "pat_bwt"             "pat_memcat_parent"  
[22] "pat_memcat_child"    "pat_memcat_subchild" "clin_discharge"      "clin_c1"             "clin_c2"             "clin_icd1"           "clin_icd2"          
[29] "clin_icd3"           "clin_icd4"           "clin_icd5"           "clin_icd6"           "clin_icd7"           "clin_icd8"           "clin_icd9"          
[36] "clin_icd10"          "clin_icd11"          "clin_icd12"          "clin_rvs1"           "clin_rvs2"           "clin_rvs3"           "clin_rvs4"          
[43] "clin_rvs5"           "clin_rvs6"        

   clin_c1_orig clin_c1
         <char>  <char>
1:        A97.1    A971
2:        I21.9    I219
3:        K29.1    K291
4:        J06.9    J069
5:        N39.0    N390
6:        P23.9    P239
   clin_c1_orig clin_c1
         <char>  <char>
1:        A97.1    A971
2:        I21.9    I219
3:        K29.1    K291
4:        J06.9    J069
5:        N39.0    N390
6:        P23.9    P239


   clin_c2_orig clin_c2
         <char>  <char>
1:        E86.1    E861
2:        I63.4    I634
3:        I61.5    I615
4:        E86.1    E861
5:        I63.9    I639
6:        E87.8    E878
   clin_c2_orig clin_c2
         <char>  <char>
1:        E86.1    E861
2:        I63.4    I634
3:        I61.5    I615
4:        E86.1    E861
5:        I63.9    I639
6:        E87.8    E878


[1] "No RVS codes discarded"
[1] "No RVS codes discarded"


[1] "No RVS codes discarded"
[1] "No RVS codes discarded"




|Column              | "" Replaced| "NA" Replaced| "character(0)" Replaced|
|:-------------------|-----------:|-------------:|-----------------------:|
|id_series           |           0|             0|                       0|
|id_pin              |           0|             0|                       0|
|date_adm            |           0|             0|                       0|
|time_adm            |           4|             0|                       0|
|date_dis            |           0|             0|                       0|
|time_dis            |           4|             0|                       0|
|date_rec            |          19|             0|                       0|
|date_ref            |      221201|             0|                       0|
|date_check          |       17295|             0|                       0|
|id_hci              |           0|             0|                       0|
|id_hcp              |         280|             0|                       0|
|pat_type 

 "Patient Types:"
[1] "DEP" "MEM"
[1] "Memcat Parent Types:"
[1] "INDIRECT" "DIRECT"   NA        
[1] "Memcat Child Types:"
[1] "INDIGENT"  "SENIOR"    "INFORMAL"  "FORMAL"    "LIFETIME"  "SPONSORED" NA         
 "Patient Types:"
[1] "DEP" "MEM"
[1] "Memcat Parent Types:"
[1] "INDIRECT" "DIRECT"   NA        
[1] "Memcat Child Types:"
[1] "INDIGENT"  "SENIOR"    "INFORMAL"  "FORMAL"    "LIFETIME"  "SPONSORED" NA         


[1] "Discharge Types:"
[1]  1  2  9  4 NA  3
[1] "Discharge Types:"
[1]  1  2  9  4 NA  3


#### Chunking

In [15]:
if (to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      num_cores <- max(1, availableCores() - 1)
      chunk_size <- ceiling(nrow(dt) / num_cores)
      chunks <- split(dt, rep(1:num_cores, each = chunk_size,
                              length.out = nrow(dt)))
      # Plan for parallel processing
      plan(multisession, workers = num_cores)
      # Process each chunk in parallel
      processed_chunks <- future_lapply(chunks, process_chunk,
                                        future.seed = global_seed)
      # Combine processed chunks
      dt <- rbindlist(processed_chunks)
      dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    num_cores <- max(1, availableCores() - 1)
    chunk_size <- ceiling(nrow(dt) / num_cores)
    chunks <- split(dt, rep(1:num_cores, each = chunk_size,
                            length.out = nrow(dt)))
    # Plan for parallel processing
    plan(multisession, workers = num_cores)
    # Process each chunk in parallel
    processed_chunks <- future_lapply(chunks, process_chunk,
                                      future.seed = global_seed)
    # Combine processed chunks
    dt <- rbindlist(processed_chunks)
    dt <- replace_empty_with_na(dt, to_view_checks)
  }
}

### Map codes and Find PDx (if not chunking)

#### Map Codes

In [16]:
#Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      dt <- map_rvs_icd9(dt, rvs_icd9)
      dt <- implement_icd10_mapping(dt)
      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    dt <- map_rvs_icd9(dt, rvs_icd9)
    dt <- implement_icd10_mapping(dt)
    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}

There are 3323 RVS codes without an ICD-9CM equivalent recognized by the TDRG ICD9CM
Of these, 1932 (100.00%) have a mapping to an ICD-9-CM code.
Of these, there are 447 (23.14%) with more than one ICD9 equivalent recognized by the Thai ICD9 library.
There are 0 (0.00%) with no ICD-9-CM equivalents.
There are 3323 RVS codes without an ICD-9CM equivalent recognized by the TDRG ICD9CM
Of these, 1932 (100.00%) have a mapping to an ICD-9-CM code.
Of these, there are 447 (23.14%) with more than one ICD9 equivalent recognized by the Thai ICD9 library.
There are 0 (0.00%) with no ICD-9-CM equivalents.


There are 7503 unique entries for ICD-10 codes, of which 4810 (64.11%) are directly in the Thai ICD-10 library
There are 7503 unique entries for ICD-10 codes, of which 4810 (64.11%) are directly in the Thai ICD-10 library


The modifications led to a total of 6357 codes being mapped to an equivalent in the Thai ICD10 library.
Out of these, 1544 were modified to match.
There are 1146 codes that could not be mapped to the Thai ICD10 library:
The modifications led to a total of 6357 codes being mapped to an equivalent in the Thai ICD10 library.
Out of these, 1544 were modified to match.
There are 1146 codes that could not be mapped to the Thai ICD10 library:


Index: <code>
        code   source
      <char>   <char>
   1:   A971 clin_icd
   2:  MCP01  clin_c2
   3:  FP001  clin_c2
   4:   A970 clin_icd
   5:  NSD01  clin_c2
  ---                
1142:   W269 clin_icd
1143:   I113 clin_icd
1144:   I571 clin_icd
1145:    189 clin_icd
1146:    X68 clin_icd
Index: <code>
        code   source
      <char>   <char>
   1:   A971 clin_icd
   2:  MCP01  clin_c2
   3:  FP001  clin_c2
   4:   A970 clin_icd
   5:  NSD01  clin_c2
  ---                
1142:   W269 clin_icd
1143:   I113 clin_icd
1144:   I571 clin_icd
1145:    189 clin_icd
1146:    X68 clin_icd


#### Find PDx

In [17]:
#Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}

In [18]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0("output_", year_to_load,
                                               suffix, ".csv")))
}

## Export

### Export for Batch Grouper

In [19]:
if (to_group) {
  if (to_profvis) {
    profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
                                  sep = "|", na.strings = "--")
      batch_grouping_result <- fread(grouper_result_file,
                                     sep = "|", na.strings = "--")
    })
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
                                sep = "|", na.strings = "--")
    batch_grouping_result <- fread(grouper_result_file,
                                   sep = "|", na.strings = "--")
  }
}

## Runtime Estimation

### Stop Timer

In [20]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic

Total execution time:: 105.81 sec elapsed
Total execution time:: 105.81 sec elapsed


### Calculate Speed

In [21]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf("Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
            formatted_total_rows_dt, total_time))
cat(sprintf("Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
            formatted_total_rows_dt, time_per_row * 1000))
cat(sprintf("Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
            formatted_total_rows, time_estimate_total_rows / 60))

Time spent (total) for 250.0k rows:  105.81 sec  (actual)
Time spent (t/row) for 250.0k rows:  0.42 msec (actual)
Time spent (total) for  11.8m rows: 83.08 min  (estimate)
Time spent (total) for 250.0k rows:  105.81 sec  (actual)
Time spent (t/row) for 250.0k rows:  0.42 msec (actual)
Time spent (total) for  11.8m rows: 83.08 min  (estimate)
